# Mancino Market: Predicting Daily Unit Sales
## Starter notebook | Business Analytics and Data Visualization
Prof. Jose Mendoza · Fall 2026

How many units will a store sell in a product category on a future day? Use the same data and evaluation process to compare KNN, a decision tree, a random forest, and a neural network. These are *regression* models: the outcome is a numeric quantity, not a class label.

Mancino Market is a synthetic grocery retailer with five fictional Manhattan stores. One row represents one date, store, and product category. Predictions can inform replenishment planning, but observed sales do not measure unmet demand during stockouts.

### How to use this notebook
1. Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/) using **File > Upload notebook**.
2. Run sections 1–5 once to prepare the data and baseline.
3. Work through sections 6–9. Change one setting at a time and rerun that model's cell. Record the validation results before changing another setting.
4. Compare models in section 10. Leave the final test switch off until you have selected your settings.
5. Complete section 11 once, then write your recommendation.

All starter code is complete. The activities focus on model settings and interpretation. No separate CSV upload is needed when the GitHub connection is available.


## 1. Import the tools
Colab includes these libraries. The notebook prints their versions so results can be traced to the environment used.


In [ ]:
import io
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


pandas: 2.2.3 | scikit-learn: 1.6.1


## 2. Load and inspect the data
Source: [mancino_daily_sales.csv](https://github.com/jrmst102/mancino/blob/main/data/v1_2025-11-01/mancino_daily_sales.csv).
The verified file contains 22,875 rows, covering January 1–November 1, 2025. The code reports the current file dimensions in case the repository changes.

| Field | Meaning | Use in this notebook |
|---|---|---|
| `date` | Selling date | Split data and derive calendar features |
| `store_id` | Store identifier | Categorical predictor |
| `category` | Product category | Categorical predictor |
| `day_of_week` | Weekday | Recomputed from the date |
| `avg_price` | Observed daily average price | Excluded |
| `units_sold` | Daily units for a store and category | Target |
| `revenue` | Realized sales revenue | Excluded |

Revenue contains information about the sales outcome and would leak information into the model. The observed average price is also excluded because its availability before the selling day has not been established. A verified, planned price could be added in a later exercise.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-11-01/mancino_daily_sales.csv'
LOCAL_CSV = Path('mancino_daily_sales.csv')

if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV)
    print('Loaded local CSV:', LOCAL_CSV)
else:
    try:
        df = pd.read_csv(DATA_URL)
        print('Loaded CSV from GitHub.')
    except Exception as error:
        print('GitHub download failed. Upload mancino_daily_sales.csv when prompted.')
        from google.colab import files
        uploaded = files.upload()
        if 'mancino_daily_sales.csv' not in uploaded:
            raise ValueError('Please upload the named Mancino CSV.') from error
        df = pd.read_csv(io.BytesIO(uploaded['mancino_daily_sales.csv']))

required = {'date', 'store_id', 'category', 'units_sold'}
assert required.issubset(df.columns), 'The CSV is missing required columns.'
df['date'] = pd.to_datetime(df['date'], errors='raise')
assert not df[list(required)].isna().any().any(), 'Check missing required values.'
assert not df.duplicated(['date', 'store_id', 'category']).any(), 'Check duplicate rows.'
assert pd.api.types.is_numeric_dtype(df['units_sold'])
assert df['units_sold'].ge(0).all(), 'Check negative unit sales.'
df = df.sort_values(['date', 'store_id', 'category']).reset_index(drop=True)
print(f'{len(df):,} rows | {df.store_id.nunique()} stores | {df.category.nunique()} categories')
print('Dates:', df.date.min().date(), 'to', df.date.max().date())
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())


Loaded CSV from GitHub.
22,875 rows | 5 stores | 15 categories
Dates: 2025-01-01 to 2025-11-01


,date,store_id,category,avg_price,units_sold,revenue,day_of_week
0,2025-01-01,1,Alcohol,12.492667,27,330.01,Wednesday
1,2025-01-01,1,Bakery,4.123889,27,113.81,Wednesday
2,2025-01-01,1,Baking,3.749231,21,73.81,Wednesday
3,2025-01-01,1,Beverages,3.329474,31,98.61,Wednesday
4,2025-01-01,1,Breakfast & Cereal,4.950833,18,89.93,Wednesday


,Missing values
date,0
store_id,0
category,0
avg_price,0
units_sold,0
revenue,0
day_of_week,0


## 3. Define predictors and split by date
Use only information available before the selling day: store, category, weekday, and day of year. Store numbers are identifiers, so they are treated as categories. Day of year supplies a simple numeric time feature; it is not a full seasonality model.

The earliest 60% of dates form the training set, the next 20% form the validation set, and the latest 20% form the test set. Every observation from a given date stays in the same set. This asks whether patterns learned from earlier dates carry forward to later dates for the existing stores and categories.

Train on the training set, choose settings using validation results, and open the test set only after those choices are complete. This starter does not use lagged sales, promotions, or inventory data.


In [ ]:
# Start the training set


In [1]:
# Explore training outcomes only; keep later outcomes for evaluation.



## 4. Prepare one shared workflow
One-hot encoding turns categories into indicator columns. Scaling puts the numeric predictor on a comparable scale for KNN and the neural network. Trees do not require scaling, but can use the same transformation.

Each model gets a fresh pipeline. Encoders and scalers are fitted on the training data only, then applied to validation data without refitting.

| Metric | Interpretation | Preferred direction |
|---|---|---|
| MAE | Average absolute error, in units per store/category/day | Lower |
| RMSE | Error in units, with larger misses receiving more weight | Lower |
| R² | Performance relative to predicting the evaluation set's mean | Higher |

R² can be negative. It is not a percentage of predictions that are correct. Use validation MAE as the selection metric for this exercise.


## 5. Establish a baseline
The baseline always predicts the training set's mean sales. A more complicated model must earn its place by improving on this simple rule. A later extension could use historical averages for each store, category, and weekday as a stronger baseline.


## 6. Exercise A: K-nearest neighbors regression
KNN predicts from the outcomes of similar training observations. Small values of *k* follow local variation closely; larger values smooth predictions.

Run the starter with `K = 15`. Then try 5 and 50, rerunning this cell each time. Compare training and validation MAE. Which setting generalizes better? Remember that similarity depends on the encoded features and their scales.


## 10. Compare validation results
Rerun this cell after changing a model. The table keeps the most recent run for each model, so record experiments in the notes below. Restore your preferred setting and rerun its model cell before selecting a finalist.

A wide gap between training and validation MAE can indicate overfitting or a change over time. Better training performance alone does not establish a better model. These calendar features are deliberately limited, and complex models may not beat the baseline.


## 11. Final test: open only after selection
The switch below is off by default. After finishing the exercises, change it to `True`. The code selects the lowest-validation-MAE model from the current comparison, refits it on training plus validation data, and evaluates it on the untouched test period. The baseline remains eligible to win.

Do not tune settings after seeing these test results. If you do, the period becomes additional validation data and you need a new test period. Predictions are left unrounded for evaluation; fractional values represent expected units.
